In [1]:
import openai, json
import requests

client = openai.OpenAI()
messages = []

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

SYSTEM_PROMPT = """
You are a movie agent.

You have access to movie tools.
Use tools when the user asks about popular movies, movie details, or similar movies.
Remember the previous conversation.
If the user asks about a movie mentioned before, use that movie's ID from the conversation.

Answer in Korean.
When you answer popular movies, include movie titles and IDs.
"""


In [2]:
def get_popular_movies():
    return requests.get(f"{BASE_URL}/movies").json()


def get_movie_details(id):
    return requests.get(f"{BASE_URL}/movies/{id}").json()


def get_similar_movies(id):
    return requests.get(f"{BASE_URL}/movies/{id}/similar").json()


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}


In [3]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get the list of popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get details of a movie by movie ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID.",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get similar movies by movie ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID.",
                    }
                },
                "required": ["id"],
            },
        },
    },
]


In [4]:
def print_tool_call(function_name, arguments):
    if arguments:
        args = ", ".join(str(value) for value in arguments.values())
        print(f"Agent: [{function_name}({args}) 호출]")
    else:
        print(f"Agent: [{function_name}() 호출]")


def process_ai_response(message):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            print_tool_call(function_name, arguments)

            function_to_run = FUNCTION_MAP.get(function_name)
            result = function_to_run(**arguments)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": json.dumps(result, ensure_ascii=False),
                }
            )

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"Agent: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            *messages,
        ],
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)


In [5]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()


User: 
Agent: 무슨 도움이 필요하신가요? 영화에 대한 질문이 있다면 언제든지 말씀해 주세요!
User: 지금 인기 있는 영화 알려줘
Agent: [get_popular_movies() 호출]
Agent: 현재 인기 있는 영화 목록입니다:

1. **Peddi**
   - ID: 1057265
   - 개요: 1980년대 안드라 프라데시 시골에서, 한 열정적인 마을 주민이 스포츠를 통해 공동체를 단합시켜 강력한 라이벌에 맞섭니다.
   - 출시일: 2026-06-03
   - 평점: 6.5
   ![Peddi](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

2. **Obsession**
   - ID: 1339713
   - 개요: 크러쉬의 마음을 얻기 위해 신비한 "원 위시 윌로우"를 깨트린 후, 희망 없는 로맨티스트가 그의 소망을 이루지만 그 대가는 어두운 가격을 치르게 됩니다.
   - 출시일: 2026-05-13
   - 평점: 7.9
   ![Obsession](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

3. **Hai Jawani Toh Ishq Hona Hai**
   - ID: 1308553
   - 개요: 결혼을 떠난 Jass가 충격적인 폭로로 인해 새로운 로맨스를 맞닥뜨리며 진정한 사랑과 충성을 다시 confront 합니다.
   - 출시일: 2026-06-04
   - 평점: 5.4
   ![Hai Jawani Toh Ishq Hona Hai](https://image.tmdb.org/t/p/w780/vmlJvz6qVzYgei2V74GvnmcuQfW.jpg)

4. **The Unknown Man**
   - ID: 879945
   - 개요: 루이, 한 플랑드르 작가가 코트 다쥐르에서 스스로를 격리하여 영감을 찾으려 합니다.
   - 출시일: 2021-10-16
   - 평점: 8.2
